# F1 NEURAL VAULT: ENTERPRISE DATA MINING PIPELINE
**Objectives:**
1. Mega-Merge all 14 historical F1 datasets into a singular dimensional model.
2. Classify drivers' probability of scoring points (Classification).
3. Cluster drivers based on race pace and pit strategy (Clustering).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, classification_report, confusion_matrix,
                             silhouette_score, davies_bouldin_score)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_theme(style="darkgrid", rc={
    "axes.facecolor": "#0B0E14", "figure.facecolor": "#0B0E14",
    "grid.color": "#1E232D", "text.color": "#E0E6ED", 
    "axes.labelcolor": "#00F2FF", "xtick.color": "#8A94A6",
    "ytick.color": "#8A94A6", "font.family": "monospace"
})

### 1. Data Ingestion (All 14 CSV Files)

In [ ]:
print("="*60)
print("INITIALIZING F1 DATA WAREHOUSE INGESTION...")
print("="*60)

circuits = pd.read_csv('circuits.csv')
constructor_results = pd.read_csv('constructor_results.csv')
constructor_standings = pd.read_csv('constructor_standings.csv')
constructors = pd.read_csv('constructors.csv')
driver_standings = pd.read_csv('driver_standings.csv')
drivers = pd.read_csv('drivers.csv')
lap_times = pd.read_csv('lap_times.csv')
pit_stops = pd.read_csv('pit_stops.csv')
qualifying = pd.read_csv('qualifying.csv')
races = pd.read_csv('races.csv')
results = pd.read_csv('results.csv')
seasons = pd.read_csv('seasons.csv')
sprint_results = pd.read_csv('sprint_results.csv')
status = pd.read_csv('status.csv')

print(f"[SUCCESS] Loaded 14 datasets. Lap Times shape: {lap_times.shape}")

### 2. Advanced Preprocessing & Feature Engineering

In [ ]:
print("\nExecuting Relational Master Merge & Feature Engineering...")

lap_agg = lap_times.groupby(['raceId', 'driverId']).agg(
    avg_lap_ms=('milliseconds', 'mean'),
    lap_variance=('milliseconds', 'std')
).reset_index()

pit_agg = pit_stops.groupby(['raceId', 'driverId']).agg(
    total_pit_stops=('stop', 'max'),
    avg_pit_duration=('milliseconds', 'mean')
).reset_index()

df = results.replace('\\N', np.nan).copy()

df = df.merge(races[['raceId', 'year', 'circuitId', 'name']], on='raceId', suffixes=('', '_race'))
df = df.merge(circuits[['circuitId', 'location', 'country']], on='circuitId')
df = df.merge(seasons[['year', 'url']], on='year', how='left')

df = df.merge(drivers[['driverId', 'dob', 'driverRef']], on='driverId')
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', suffixes=('', '_team'))
df = df.merge(status[['statusId', 'status']], on='statusId')

df = df.merge(driver_standings[['raceId', 'driverId', 'points', 'position']], on=['raceId', 'driverId'], suffixes=('', '_championship'))
df = df.merge(constructor_standings[['raceId', 'constructorId', 'points']], on=['raceId', 'constructorId'], suffixes=('', '_constructor_champ'))

df = df.merge(lap_agg, on=['raceId', 'driverId'], how='left')
df = df.merge(pit_agg, on=['raceId', 'driverId'], how='left')
df = df.merge(qualifying[['raceId', 'driverId', 'position']], on=['raceId', 'driverId'], how='left', suffixes=('', '_quali'))

columns_to_numeric = ['grid', 'positionOrder', 'points', 'fastestLapSpeed', 'position_quali']
for col in columns_to_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['Scored_Points'] = df['positionOrder'].apply(lambda x: 1 if x <= 10 else 0)

df_clean = df.dropna(subset=['lap_variance', 'avg_pit_duration', 'grid', 'fastestLapSpeed']).copy()
df_clean = df_clean[df_clean['avg_pit_duration'] < 60000]

print(f"[SUCCESS] Master Database Assembled. Final Shape: {df_clean.shape}")

### 3. Descriptive Statistics & Visualizations

In [ ]:
print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS MATRIX")
print("="*60)
desc_stats = df_clean[['grid', 'lap_variance', 'avg_pit_duration', 'fastestLapSpeed', 'points_championship']].describe().round(2)
print(desc_stats)

print("\nRendering Multi-Dimensional Analytics Dashboard...")
fig, axes = plt.subplots(2, 2, figsize=(22, 16))
fig.suptitle('F1 DATA MINING: DESCRIPTIVE VISUAL SUMMARIES', fontsize=24, color='#00F2FF', fontweight='bold', y=0.98)

sns.kdeplot(ax=axes[0, 0], data=df_clean, x='lap_variance', hue='Scored_Points', fill=True, palette=['#FF0055', '#00F2FF'], alpha=0.5)
axes[0, 0].set_title('DRIVER CONSISTENCY: LAP VARIANCE DISTRIBUTION', color='#FFD700', fontsize=14)
axes[0, 0].set_xlim(0, 150000)
axes[0, 0].set_xlabel('Lap Time Variance (ms)')

sns.scatterplot(ax=axes[0, 1], data=df_clean, x='position_quali', y='positionOrder', hue='Scored_Points', palette='viridis', alpha=0.6, s=60)
axes[0, 1].set_title('QUALIFYING POSITION VS FINAL RACE POSITION', color='#FFD700', fontsize=14)
axes[0, 1].plot([0, 25], [0, 25], color='#FF0055', linestyle='--', linewidth=2)

top_teams = df_clean['name_team'].value_counts().head(6).index
sns.boxplot(ax=axes[1, 0], data=df_clean[df_clean['name_team'].isin(top_teams)], x='name_team', y='avg_pit_duration', palette='mako')
axes[1, 0].set_title('PIT CREW EFFICIENCY BY TOP CONSTRUCTORS', color='#FFD700', fontsize=14)
axes[1, 0].set_ylim(20000, 45000)
axes[1, 0].set_ylabel('Avg Pit Duration (ms)')

corr_cols = ['grid', 'position_quali', 'lap_variance', 'avg_pit_duration', 'fastestLapSpeed', 'points_championship', 'Scored_Points']
sns.heatmap(ax=axes[1, 1], data=df_clean[corr_cols].corr(), annot=True, cmap='inferno', fmt=".2f", linewidths=1, linecolor='#0B0E14')
axes[1, 1].set_title('TELEMETRY & RESULTS CORRELATION MATRIX', color='#FFD700', fontsize=14)

plt.tight_layout(pad=3.0)
plt.show()

### 4. Classification: Random Forest

In [ ]:
print("\n" + "="*60)
print("TECHNIQUE 1: RANDOM FOREST CLASSIFICATION")
print("Objective: Predict Points Finish (Top 10) based on Telemetry & Grid")
print("="*60)

features_clf = ['grid', 'position_quali', 'lap_variance', 'avg_pit_duration', 'total_pit_stops', 'fastestLapSpeed']
X_clf = df_clean[features_clf].fillna(df_clean[features_clf].median())
y_clf = df_clean['Scored_Points']

X_train, X_test, y_train, y_test = train_test_split(X_clf, y_clf, test_size=0.25, random_state=42, stratify=y_clf)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_model = RandomForestClassifier(n_estimators=250, max_depth=15, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred = rf_model.predict(X_test_scaled)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap="Blues", cbar=False, annot_kws={"size": 14})
plt.title("CLASSIFICATION: CONFUSION MATRIX", color="#00F2FF", pad=15)
plt.xlabel("Predicted (0 = No Points, 1 = Points)")
plt.ylabel("Actual")
plt.show()

### 5. Clustering: K-Means

In [ ]:
print("\n" + "="*60)
print("TECHNIQUE 2: K-MEANS CLUSTERING")
print("Objective: Segment race strategies based on Pace vs. Pit Durations")
print("="*60)

features_clust = ['fastestLapSpeed', 'avg_pit_duration', 'lap_variance']
X_clust = df_clean[features_clust].dropna()

robust_scaler = RobustScaler()
X_clust_scaled = robust_scaler.fit_transform(X_clust)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_clust_scaled)
X_clust['Strategy_Cluster'] = cluster_labels

sil_score = silhouette_score(X_clust_scaled, cluster_labels)
dbi_score = davies_bouldin_score(X_clust_scaled, cluster_labels)

print(f"Optimal Clusters: 3")
print(f"Silhouette Score (Cohesion/Separation): {sil_score:.4f}")
print(f"Davies-Bouldin Index (Cluster Similarity): {dbi_score:.4f}")

plt.figure(figsize=(12, 7))
scatter = sns.scatterplot(x='fastestLapSpeed', y='avg_pit_duration', hue='Strategy_Cluster', 
                          size='lap_variance', sizes=(20, 300), palette=['#FFD700', '#00F2FF', '#FF0055'], 
                          data=X_clust, alpha=0.7, edgecolor='#0B0E14')
plt.title("STRATEGY CLUSTERING: PACE vs PIT DURATION vs CONSISTENCY", color="#00F2FF", fontweight='bold', fontsize=14)
plt.xlabel("Fastest Lap Speed (km/h)")
plt.ylabel("Average Pit Stop Duration (ms)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("DATA MINING PIPELINE EXECUTION COMPLETE.")
print("="*60)